# Institutional ownership from 13F filings, 2010Q1 to 2014Q4

Luke Freyman, for Dr. Akcora / Voloridge.

**Ownership % = shares held by all 13F filers / shares outstanding**, per company and quarter.

Steps: 1. filing index, 2. parse 13Fs, 3. match CUSIP to company, 4. shares outstanding, 5. ownership, 6. validation, then the CSV and a data-quality summary.

Runs offline from `cache/` (`EDGAR_OFFLINE=1`). Run: `jupyter nbconvert --to notebook --execute --inplace notebook.ipynb`

## Configuration

`QUARTERS` sets which quarters run (default: the two pilots; `quarters.ALL_LABELS` = all 20).

In [1]:
import io, os, sys, subprocess, contextlib
from pathlib import Path
import pandas as pd

QUARTERS = ["pilot_2013Q4", "pilot_2010Q1"]      # <- edit this list to run other quarters (quarters.ALL_LABELS = all 20)
WORKERS = 4                                      # parser processes (Luke's cap while other jobs run)

ROOT = Path.cwd() if (Path.cwd() / "pipeline").is_dir() else Path.cwd().parent
os.chdir(ROOT)
os.environ["EDGAR_OFFLINE"] = "1"                # cache only: any attempt to download raises an error
os.environ["EDGAR_WORKERS"] = str(WORKERS)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

from pipeline import quarters, index as indexmod, run as runmod, shares_check, cusip_cik_resolve
from pipeline import shares_out, shares_out_regex, ownership, validate_secds
from pipeline import config as cfgmod
from pipeline.edgar import Edgar

def quiet(fn, *args, **kw):
    """Run a pipeline function, keep its printed report as text instead of flooding the notebook."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        result = fn(*args, **kw)
    return result, buf.getvalue()

def script(name, *args):
    """Run one of the report scripts in scripts/ (they write to output/problems/<label>/) and return its stdout."""
    r = subprocess.run([sys.executable, f"scripts/{name}.py", *map(str, args)], capture_output=True, text=True, env=os.environ)
    if r.returncode != 0:
        raise RuntimeError(f"scripts/{name}.py {' '.join(map(str, args))} failed:\n{r.stderr[-2000:]}")
    return r.stdout

def lines(text, *keys, n=None):
    """Print only the report lines that contain one of the keys (or the first n lines)."""
    out = [l for l in text.splitlines() if not keys or any(k in l for k in keys)]
    print("\n".join(out[:n] if n else out))

def report(label, name):
    return (Path("output/problems") / label / name).read_text()

SPECS = {q: quarters.spec(q) for q in QUARTERS}
edgar = Edgar(cfgmod.load())
blocked = {}
for q, s in SPECS.items():
    miss = quarters.missing(s)
    optional = {k for k in miss if k.startswith("SEC 13F data set")}
    print(f"{q}: EDGAR index {s['index_year']}Q{s['index_qtr']}, quarter end {s['quarter_end']}, 13G/13D links {s['sc13_label']}, "
          f"XBRL notes {[Path(z).stem for z in s['notes_zips']]}")
    for k, how in miss.items():
        print(f"    {'optional' if k in optional else 'MISSING '} {k}: {how}")
    if set(miss) - optional:
        blocked[q] = set(miss) - optional
assert not blocked, f"cached inputs missing for {sorted(blocked)}: run the commands above first (they need the network)"
RESULTS = {}

pilot_2013Q4: EDGAR index 2014Q1, quarter end 2013-12-31, 13G/13D links sc13_2013, XBRL notes ['2014q1_notes', '2014q2_notes']
pilot_2010Q1: EDGAR index 2010Q2, quarter end 2010-03-31, 13G/13D links sc13_2010, XBRL notes ['2010q2_notes', '2010q3_notes']
    optional SEC 13F data set (validation only): not published by the SEC for this quarter


## 1. Filing index

Every 13F-HR and 13F-HR/A for the quarter from the SEC index. 13F-NT notices are skipped.

In [2]:
rows = []
for q, s in SPECS.items():
    idx = indexmod.filings_13f(edgar, s["index_year"], s["index_qtr"])
    rows.append({"quarter": q, "EDGAR index": f"{s['index_year']}Q{s['index_qtr']}", "13F-HR": int((idx["form_type"] == "13F-HR").sum()),
                 "13F-HR/A": int((idx["form_type"] == "13F-HR/A").sum()), "filings": len(idx), "distinct managers": idx["cik"].nunique()})
pd.DataFrame(rows).set_index("quarter")

,EDGAR index,13F-HR,13F-HR/A,filings,distinct managers
quarter,,,,,
pilot_2013Q4,2014Q1,3777,263,4040,3707
pilot_2010Q1,2010Q2,2930,456,3386,2900


## 2. Parse 13Fs

Each filing's holdings (XML, HTML or plain text) become rows: fund, CUSIP, shares, value.

**Checksum:** parsed row count and total value must match the filing's own summary page. Failures are kept and labeled with a cause.

**Price check:** value / shares is compared with other filers of the same security to catch unit and column errors. Reported values are never changed.

In [3]:
for q, s in SPECS.items():
    print(f"=== {q} ===")
    (filings, holdings), log = quiet(runmod.run_quarter, s["index_year"], s["index_qtr"], q)
    lines(log, "parse done", "checksum:", "cusip_status rows")
    _, txt = quiet(shares_check.run, q)
    lines(txt, "PASS 2")
    # standard per-quarter reports; each writes a file under output/problems/<quarter>/
    for name, args in [("failure_accounting", ()), ("triage_checksum", ()), ("big_filers", (25,)), ("weighted_pass", ())]:
        script(name, q, *args)
    lines(report(q, "failure_accounting.txt"), "over ALL", "holdings-bearing filings:", "usable-summary filings:")
    print()

=== pilot_2013Q4 ===


14:27:51 parse done: 4040 filings, 1584855 holdings rows
14:27:52 checksum: pass=3660, fail_entries=168, fail_value=123, fail_both=87, no_summary=2
14:27:55 cusip_status rows: ok=1556120, option_code=13053, repaired_check_digit=8304, repaired_pad_zeros=2364, sedol=1834, invalid_9=1616, ambiguous=1058, unconfirmed_short=506
entry-count checksum: summary present 4038/4038; pass 3783/4038 = 93.7%
value-total checksum: summary present 4038/4038; pass 3828/4038 = 94.8%


  over ALL 4040 index filings:                     pass  3660  = 90.6%   (both totals match:  3660 = 90.6%)
  over 4038 holdings-bearing filings:          pass  3660  = 90.6%   (both totals match:  3660 = 90.6%)
  over 3973 usable-summary filings:            pass  3653  = 91.9%   (both totals match:  3653 = 91.9%)

=== pilot_2010Q1 ===


14:28:36 parse done: 3386 filings, 1413090 holdings rows
14:28:36 checksum: pass=2207, fail_both=354, fail_value=314, no_summary=301, fail_entries=210
14:28:39 cusip_status rows: ok=1358765, repaired_check_digit=35619, option_code=10424, sedol=2417, repaired_pad_zeros=2035, unconfirmed_short=1743, invalid_9=1606, ambiguous=481
entry-count checksum: summary present 3063/3377; pass 2499/3063 = 81.6%
value-total checksum: summary present 3033/3377; pass 2365/3033 = 78.0%


  over ALL 3386 index filings:                     pass  2207  = 65.2%   (both totals match:  2163 = 63.9%)
  over 3377 holdings-bearing filings:          pass  2207  = 65.4%   (both totals match:  2163 = 64.1%)
  over 2974 usable-summary filings:            pass  2141  = 72.0%   (both totals match:  2141 = 72.0%)



In [4]:
# Parse-method mix and the mutually exclusive failure classes, straight from the accounting report
for q in SPECS:
    fa = report(q, "failure_accounting.txt")
    start = fa.index("Holdings-bearing filings, one bucket each"); end = fa.index("Summary page presence")
    print(f"--- {q}: parse methods " + pd.read_parquet(f"cache/parsed/{q}_filings.parquet", columns=["parse_method"])["parse_method"].value_counts().to_dict().__str__())
    print(fa[start:end].rstrip())

--- pilot_2013Q4: parse methods {'xml': 4040}
Holdings-bearing filings, one bucket each (checksum_cause from pipeline/checksum.py):
   3660  pass: all present totals match
     80  entry total counts distinct CUSIPs not rows
     78  value scale x1000 (dollars vs thousands)
     75  entry count off by >2, unexplained
     42  empty filing (single placeholder row, totals zero)
     36  value off >1%, unexplained
     28  entry count off by <=2
     17  value within 1% (rounding)
     14  summary totals zero but holdings present (filer omitted totals)
      4  entry off by <=2 and value within 1%
      2  summary fields swapped (entry total = value total)
      2  entry off by <=2 and value off >1%
  -----
   4038  == 4038
--- pilot_2010Q1: parse methods {'text': 3383, 'html': 3}
Holdings-bearing filings, one bucket each (checksum_cause from pipeline/checksum.py):
   2207  pass: all present totals match
    292  summary page missing
    247  entry count off by >2, unexplained
    215  va

## 3. Match CUSIP to company

13Fs list CUSIPs; share counts are filed under company CIKs. Links come from 13G/13D cover pages, then exact names on the SEC 13(f) list. Conflicts are flagged, never guessed (stop rule: 2%).

In [5]:
for q, s in SPECS.items():
    print(f"=== {q} ===")
    _, txt = quiet(cusip_cik_resolve.resolve, q, s["sc13_label"], s["quarter_end"], s["list_year"], s["list_qtr"])
    lines(txt, "valid 9-char CUSIPs", "conflicts, raw", "TWO-SIDED", "exact unique name matches added", "coverage of 13F value")
    print()

=== pilot_2013Q4 ===


valid 9-char CUSIPs with a subject CIK: 6,096 (placeholder 000000000 dropped from 13 filings)
conflicts, raw definition (>1 subject CIK): 220 = 3.61%  -> all flagged conflict_flag=1
conflicts, TWO-SIDED definition (>=2 filings on each of >=2 CIKs): 27 = 0.44%   [stop condition 2%: not hit]
exact unique name matches added: 593 issuers = $953.2bn; of these fund CIKs (excluded later): 48 = $281.5bn
coverage of 13F value, all issuers: 13G/13D map 93.3% -> with name matches 97.5%

=== pilot_2010Q1 ===


valid 9-char CUSIPs with a subject CIK: 6,711 (placeholder 000000000 dropped from 3 filings)
conflicts, raw definition (>1 subject CIK): 211 = 3.14%  -> all flagged conflict_flag=1
conflicts, TWO-SIDED definition (>=2 filings on each of >=2 CIKs): 24 = 0.36%   [stop condition 2%: not hit]
exact unique name matches added: 651 issuers = $1,027.7bn; of these fund CIKs (excluded later): 44 = $138.1bn
coverage of 13F value, all issuers: 13G/13D map 89.6% -> with name matches 96.6%



## 4. Shares outstanding

XBRL cover-page share count closest to quarter end. Fallback: 10-K cover text (only within 45 days of filing). Share classes keep separate counts. 2010 has gaps because XBRL was still phasing in.

In [6]:
for q, s in SPECS.items():
    print(f"=== {q} ===")
    _, txt = quiet(shares_out.build, q, s["quarter_end"], s["notes_zips"])
    lines(txt, n=3)
    _, txt = quiet(shares_out_regex.run, q, s["quarter_end"])
    lines(txt, "regex captured", "attempt status")
    print()

=== pilot_2013Q4 ===


mapped operating companies 4,951; missing an XBRL fact 603; regex captured 8 = 1.3% of missing
attempt status: {'no 10-K filed within the window': 564, 'no match': 29, 'ok': 8, 'as-of more than 45 days from filing date': 2}

=== pilot_2010Q1 ===


mapped operating companies 5,245; missing an XBRL fact 3,937; regex captured 1,249 = 31.7% of missing
attempt status: {'no match': 1359, 'ok': 1249, 'no 10-K filed within the window': 1247, 'as-of more than 45 days from filing date': 82}



### Placeholder: Dr. Akcora's 2010 shares outstanding

Expected at `data/akcora_shares_outstanding_2010.csv` (cusip or cik, shares_outstanding, as_of_date, source). This cell checks the file only; merging waits on the agreed rule.

In [7]:
AKCORA_SHARES_FILE = Path("data/akcora_shares_outstanding_2010.csv")      # <- expected location of the provided file
AKCORA_KEY_COLUMNS = {"cusip", "cik"}                                      # one of these
AKCORA_VALUE_COLUMNS = {"shares_outstanding", "as_of_date", "source"}

if not AKCORA_SHARES_FILE.exists():
    print(f"PLACEHOLDER: {AKCORA_SHARES_FILE} not provided yet; the ownership step below uses XBRL + the 10-K regex fallback only.")
else:
    ak = pd.read_csv(AKCORA_SHARES_FILE, dtype={"cusip": str})
    key = AKCORA_KEY_COLUMNS & set(ak.columns)
    assert key and AKCORA_VALUE_COLUMNS <= set(ak.columns), f"expected a cusip or cik column plus {sorted(AKCORA_VALUE_COLUMNS)}; got {list(ak.columns)}"
    ak["as_of_date"] = pd.to_datetime(ak["as_of_date"])
    print(f"{AKCORA_SHARES_FILE}: {len(ak):,} rows keyed by {sorted(key)}, as-of dates {ak['as_of_date'].min().date()} to {ak['as_of_date'].max().date()}")
    for q, s in SPECS.items():
        if s["period_year"] != 2010:
            continue
        so = pd.read_parquet(f"cache/parsed/{q}_shares_out.parquet")
        m = pd.read_csv(f"data/{q}_cusip_cik_map.csv", dtype={"cusip6": str, "cusip9": str})
        have = set(so["cik"]); mapped = set(m["company_cik"].dropna().astype(int))
        his = set(ak["cik"].dropna().astype(int)) if "cik" in key else set(m.loc[m["cusip6"].isin(ak["cusip"].str[:6]), "company_cik"].dropna().astype(int))
        print(f"  {q}: mapped companies {len(mapped):,}; with an XBRL/regex figure {len(mapped & have):,}; his file would add {len((mapped - have) & his):,} of the {len(mapped - have):,} still missing")
    print("NOT merged into the ownership step: the merge rule is not yet agreed.")

PLACEHOLDER: data/akcora_shares_outstanding_2010.csv not provided yet; the ownership step below uses XBRL + the 10-K regex fallback only.


## 5. Ownership

Amendments resolved (restatement replaces, new holdings adds). Shares-only rows, no options. Shares summed per CUSIP across filers, divided by shares outstanding.

Funds and ETFs excluded. Debt CUSIPs and ADRs flagged. Classes never summed. Over 100% kept and flagged with a cause, not capped.

In [8]:
for q, s in SPECS.items():
    print(f"=== {q} ===")
    (comp, mc), txt = quiet(ownership.run, q, s["quarter_end"], s["period"])
    RESULTS[q] = mc
    lines(txt, "after amendment/duplicate resolution", "holdings rows used", "FUNDS EXCLUDED", "FIXES (Luke", "SHARE CLASSES",
          "coverage among OPERATING", "shares outstanding (", "companies over 100%", "large companies (top 500")
    print(f"full report: output/problems/{q}/ownership_report.txt; rows in the output table: {len(mc):,}")
    print()

=== pilot_2013Q4 ===


pipeline/funds.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = s["issuer_name"].str.contains(F3_TOKENS) | s["issuer_desc"].str.contains(F3_TOKENS) | s["issuer_desc"].str.contains(F3_DESC)
pipeline/funds.py:63: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  s = seclist[seclist["issuer_name"].str.contains(N_TOKENS)].drop_duplicates("cusip6").set_index("cusip6")


filings for the period: 3,885; after amendment/duplicate resolution: 3,733 ({'13F-HR': 3560, '13F-HR/A': 173}); notes: 3 (first 5 below)
holdings rows used: 1,289,567 (SH, no put/call, resolved CUSIP, shares > 0)
FUNDS EXCLUDED (decision 2): 1,883 CUSIPs on 566 CIKs, 89,142 manager-CUSIP rows, $939.8bn of 13F value = 4.7% of the selected filings' value
FIXES (Luke 2026-10-12): debt-pattern CUSIPs excluded from ownership (rows kept): 2,108 rows / 1,057 CUSIPs / $71.6bn; ADR/ADS excluded from the main figure: 39,363 rows / 379 CUSIPs on 283 companies / $589.3bn; NEW HOLDINGS amendments duplicating >= 90% of the original treated as restatements: 14 amendments, 14 original filings / 5,134 rows dropped
SHARE CLASSES (never summed): multi-class companies (class-dimensioned facts only) 324; their CUSIPs: matched to a class 234 ($1,307.9bn), unmatched 177 ($359.6bn); company-level pct is null for all 324
CUSIP->CIK coverage among OPERATING companies (6-char issuers, funds removed: 724 incl. 11

pipeline/funds.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = s["issuer_name"].str.contains(F3_TOKENS) | s["issuer_desc"].str.contains(F3_TOKENS) | s["issuer_desc"].str.contains(F3_DESC)
pipeline/funds.py:63: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  s = seclist[seclist["issuer_name"].str.contains(N_TOKENS)].drop_duplicates("cusip6").set_index("cusip6")


filings for the period: 3,006; after amendment/duplicate resolution: 2,863 ({'13F-HR': 2742, '13F-HR/A': 121}); notes: 134 (first 5 below)
holdings rows used: 1,034,768 (SH, no put/call, resolved CUSIP, shares > 0)
FUNDS EXCLUDED (decision 2): 1,488 CUSIPs on 535 CIKs, 46,165 manager-CUSIP rows, $649.6bn of 13F value = 5.1% of the selected filings' value
FIXES (Luke 2026-10-12): debt-pattern CUSIPs excluded from ownership (rows kept): 5,309 rows / 3,869 CUSIPs / $22.1bn; ADR/ADS excluded from the main figure: 34,181 rows / 366 CUSIPs on 262 companies / $543.1bn; NEW HOLDINGS amendments duplicating >= 90% of the original treated as restatements: 0 amendments, 0 original filings / 0 rows dropped
SHARE CLASSES (never summed): multi-class companies (class-dimensioned facts only) 58; their CUSIPs: matched to a class 50 ($406.2bn), unmatched 80 ($115.9bn); company-level pct is null for all 58
CUSIP->CIK coverage among OPERATING companies (6-char issuers, funds removed: 654 incl. 91 unmapped 

## 6. Validation

1. Parse vs the SEC's own 13F data set (2013Q2 on).
2. Checksum pass rates, all 20 quarters.
3. Ownership distribution and every company over 100%.
4. Hand check of the 20 largest companies.

In [9]:
# 6.1 our parse vs the SEC structured 13F data set (2013Q2 onward; the data-set zip must be in the cache)
for q, s in SPECS.items():
    if not s["secds_available"]:
        print(f"{q}: the SEC 13F data set starts in 2013Q2; not available for this quarter\n"); continue
    if not Path(s["inputs"]["SEC 13F data set (validation only)"]).exists():
        print(f"{q}: SEC 13F data set for index {s['index_year']}Q{s['index_qtr']} is not in the cache (optional input)\n"); continue
    txt = validate_secds.run(q, s["index_year"], s["index_qtr"])
    print(f"=== {q} vs SEC data set {s['index_year']}q{s['index_qtr']} ===")
    lines(txt, n=4)
    print()

=== pilot_2013Q4 vs SEC data set 2014q1 ===
our filings: 4040   in SEC data set: 4038   ours not in SEC DS: 2   SEC DS not in ours: 1
filing-level exact match — rows: 100.00%   shares: 100.00%   value: 100.00%
row-level exact match on (cusip, shares, value): 1584855/1584855 = 100.000%
filings matching on all three: 4038/4038 = 100.00%

pilot_2010Q1: the SEC 13F data set starts in 2013Q2; not available for this quarter



In [10]:
# 6.2 summary-page checksum pass rates by quarter (all twenty, round 3b of 2026-10-12; rows re-run in this notebook are marked)
tab = pd.read_csv("output/problems/all_quarters_table.csv", index_col="quarter")
cols = ["filings", "holdings_bearing", "text%", "xml%", "pass_filings%", "pass_usable%", "pass+classA_plausible%", "shares_flag%", "no_rows", "x1000_filings"]
view = tab[cols].copy()
view.insert(0, "re-run here", ["yes" if q in SPECS else "" for q in view.index])
print(f"lowest pass+classA_plausible%: {tab['pass+classA_plausible%'].idxmin()} = {tab['pass+classA_plausible%'].min()}; quarters under 85%: {int((tab['pass+classA_plausible%'] < 85).sum())}")
view

lowest pass+classA_plausible%: 2011Q3 = 86.3; quarters under 85%: 0


,re-run here,filings,holdings_bearing,text%,xml%,pass_filings%,pass_usable%,pass+classA_plausible%,shares_flag%,no_rows,x1000_filings
quarter,,,,,,,,,,,
pilot_2010Q1,yes,3386,3377,99.9,0.0,65.4,72.0,94.5,3.53,72,199
2010Q2,,3191,3182,100.0,0.0,67.3,74.9,93.1,2.08,69,222
2010Q3,,3161,3148,99.9,0.0,67.8,74.6,93.3,2.79,73,183
2010Q4,,3591,3576,99.9,0.0,69.1,75.7,91.4,4.64,79,188
2011Q1,,3585,3577,99.9,0.0,68.4,74.8,90.7,4.03,64,176
2011Q2,,3496,3490,99.9,0.0,69.1,75.1,86.5,2.30,83,171
2011Q3,,3483,3477,99.9,0.0,71.5,77.2,86.3,5.10,80,165
2011Q4,,3679,3672,100.0,0.0,72.1,77.5,92.7,4.75,75,168
2012Q1,,3615,3610,99.9,0.0,71.4,76.6,93.0,4.42,63,172


In [11]:
# 6.3 ownership distribution and the over-100% companies with causes (ownership_report.txt, over_100_companies.csv)
for q in SPECS:
    print(f"=== {q} ===")
    lines(report(q, "ownership_report.txt"), "ownership pct distribution", "histogram:", "companies over 100%", "large companies (top 500")
    over = pd.read_csv(f"output/problems/{q}/over_100_companies.csv")
    print(f"over-100% companies by cause ({len(over):,} companies):")
    print(over["over_100_cause"].str.split(" [", regex=False).str[0].value_counts().to_string())
    print()

=== pilot_2013Q4 ===
ownership pct distribution (n=3,937): {'count': 3937.0, 'mean': 16064.4, 'std': 1002404.4, 'min': 0.0, '5%': 1.4, '25%': 28.7, '50%': 65.0, '75%': 87.5, '95%': 101.5, 'max': 62896403.5}
histogram: {'0': 0, '0-5': 334, '5-25': 552, '25-50': 643, '50-75': 813, '75-90': 766, '90-100': 588, '100-150': 230, '>150': 11}
companies over 100%: 241 / 3,937 = 6.1%
large companies (top 500 by 13F value) with a pct: 419; over 100%: 28 = 6.7%  [stop condition at 5%]
over-100% companies by cause (241 companies):
over_100_cause
no identified data cause;                                                                                                                                   115
manager-CUSIP in >1 filing;                                                                                                                                  58
multiple CUSIPs summed (classes/units);                                                                                                      

In [12]:
# 6.4 hand check of the 20 largest companies: every input shown (full file: output/problems/<quarter>/hand_check.txt)
for q in SPECS:
    script("hand_check", q, 20)
    hc = report(q, "hand_check.txt")
    blocks = hc.split("=" * 110)
    print(blocks[0].rstrip())
    print(("=" * 110).join(blocks[1:3]).rstrip())          # the two largest companies in full
    print(f"... {len(blocks) - 1} companies in output/problems/{q}/hand_check.txt\n")

pilot_2013Q4: hand check of the 20 largest companies by 13F value (every input shown)

APPLE INC   CIK 320193   13F value $301.1bn   map_source SC 13G   conflict 0/0
  CUSIPs held (counted):
           managers  filings     shares       value_k desc  class_shares_out  class_pct   class_match  median_px
cusip                                                                                                           
037833100      1881     1883  536126739  3.011132e+08  COM       891989000.0  60.104636  single-class     561.02
  shares held total 536,126,739  managers 1881  filings 1883  multi_filing rows 4
  shares outstanding facts (XBRL, all rows for the CIK):
class_axis      shares form                 adsh shares_outstanding_date      filed  gap_days  gap_flag
           891989000.0 10-Q 0001193125-14-024487              2013-12-31 2014-01-28         0     False
  USED: shares_outstanding 891,989,000  source 10-Q 0001193125-14-024487  as-of 2013-12-31  multi_class False
  ==> institu

pilot_2010Q1: hand check of the 20 largest companies by 13F value (every input shown)

MICROSOFT CORP   CIK 789019   13F value $204.6bn   map_source SC 13G   conflict 0/0
  CUSIPs held (counted):
           managers  filings      shares       value_k desc  class_shares_out  class_pct   class_match  median_px
cusip                                                                                                            
594918104      1594     1594  5251508332  2.044333e+08  COM      8.763839e+09  59.922462  single-class      29.29
594918906         1        1     2594400  7.598500e+04           8.763839e+09   0.029603  single-class      29.29
594918955         1        1     2279400  6.675900e+04           8.763839e+09   0.026009  single-class      29.29
  CUSIPs EXCLUDED from the figure:
                                             managers   shares
cusip     excluded_from_ownership list_desc                   
594918AB0 debt_cusip                                1  1030000
594918AC8 

## Final output

One CSV per quarter, one row per fund, security and quarter. First 12 columns follow the spec; the rest are flags and sources. Nothing is dropped, only flagged.

In [13]:
written = []
for q in SPECS:
    p = ownership.write_output(RESULTS[q], q)
    chk = pd.read_csv(p, nrows=0)
    assert list(chk.columns) == ownership.OUTPUT_COLUMNS, f"{p}: column order differs from the spec"
    written.append({"quarter": q, "file": str(p), "rows": len(RESULTS[q]), "companies": RESULTS[q]["company_cik"].nunique(),
                    "managers": RESULTS[q]["manager_cik"].nunique(), "size_MB": round(p.stat().st_size / 1e6, 1)})
print("spec columns (12):", ownership.REQUIRED)
print("traceability columns:", ownership.TRACEABILITY)
pd.DataFrame(written).set_index("quarter")

spec columns (12): ['quarter_end', 'company_name', 'company_cik', 'cusip', 'manager_cik', 'manager_name', 'shares_held', 'shares_outstanding', 'institutional_ownership_pct', 'form_13f_accession_number', 'shares_outstanding_source', 'shares_outstanding_date']
traceability columns: ['shares_outstanding_evidence', 'cusip_issuer', 'map_source', 'list_desc', 'excluded_from_ownership', 'class_shares_outstanding', 'class_ownership_pct', 'class_match', 'multi_class_flag', 'map_conflict', 'map_conflict_two_sided', 'multi_filing_flag', 'dup_amendment_flag', 'debt_cusip_flag', 'adr_flag', 'over_100_flag', 'over_100_cause', 'n_rows']


,file,rows,companies,managers,size_MB
quarter,,,,,
pilot_2013Q4,output/pilot_2013Q4_institutional_ownership.csv,829798,4911,3604,233.4
pilot_2010Q1,output/pilot_2010Q1_institutional_ownership.csv,662004,5184,2766,196.5


## 7. Data-quality summary

Counts below come from `output/problems/<quarter>/`.

In [14]:
HANDLING = {
    # checksum classes (pipeline/checksum.py), mutually exclusive, one per holdings-bearing filing
    "entry total counts distinct CUSIPs not rows": "filer declared the number of securities, not rows; rows kept, filing marked class (a) 'parse right, summary wrong'",
    "value scale x1000 (dollars vs thousands)": "filer reported dollars; value_scale_flag=x1000 on the filing, reported value never overwritten, shares used for ownership",
    "entry count off by >2, unexplained": "rows kept; failure logged with the declared and parsed counts for the triage and big-filer reviews",
    "entry count off by <=2": "rows kept; a one-or-two row gap (a total line or a footnote) is logged, not fixed by hand",
    "empty filing (single placeholder row, totals zero)": "filer reported no holdings ('No Issuers to Report'); no rows, filing counted as a failure for honesty",
    "value off >1%, unexplained": "rows kept; logged, value never adjusted",
    "value within 1% (rounding)": "rows kept; filer rounding; logged",
    "summary totals zero but holdings present (filer omitted totals)": "rows kept; no total to check against; logged",
    "entry off by <=2 and value within 1%": "rows kept; logged",
    "entry off by <=2 and value off >1%": "rows kept; logged",
    "summary fields swapped (entry total = value total)": "filer put the totals in each other's boxes; rows kept; logged",
    "no rows parsed": "text layout the parser cannot read; filing carries no holdings in the output; listed for the next fixture round",
}
def classes(q):
    pr = pd.read_csv(f"output/problems/{q}/problems.csv")
    return pr[pr["stage"] == "checksum"].groupby("reason").size().rename(q)       # one bucket per holdings-bearing filing
tab = pd.concat([classes(q) for q in SPECS], axis=1).fillna(0).astype(int)
tab.index.name = "problem class (checksum stage, mutually exclusive, one per failing filing)"
tab["handled by"] = [HANDLING.get(k, "") for k in tab.index]
for q in SPECS:
    pr = pd.read_csv(f"output/problems/{q}/problems.csv")
    print(f"{q}: parse-stage entries (text filings with no holdings read) {int((pr['stage'] == 'parse').sum())}; "
          f"these filings are also in a checksum bucket below, so they are not added to the table")
print("Filings per problem class (summary-page mismatch classes from pipeline/checksum.py)")
with pd.option_context("display.max_colwidth", None):
    display(tab.sort_values(list(SPECS), ascending=False))

pilot_2013Q4: parse-stage entries (text filings with no holdings read) 0; these filings are also in a checksum bucket below, so they are not added to the table
pilot_2010Q1: parse-stage entries (text filings with no holdings read) 81; these filings are also in a checksum bucket below, so they are not added to the table
Filings per problem class (summary-page mismatch classes from pipeline/checksum.py)


,pilot_2013Q4,pilot_2010Q1,handled by
"problem class (checksum stage, mutually exclusive, one per failing filing)",,,
entry total counts distinct CUSIPs not rows,80,141,"filer declared the number of securities, not rows; rows kept, filing marked class (a) 'parse right, summary wrong'"
value scale x1000 (dollars vs thousands),78,215,"filer reported dollars; value_scale_flag=x1000 on the filing, reported value never overwritten, shares used for ownership"
"entry count off by >2, unexplained",75,247,rows kept; failure logged with the declared and parsed counts for the triage and big-filer reviews
"empty filing (single placeholder row, totals zero)",42,7,"filer reported no holdings ('No Issuers to Report'); no rows, filing counted as a failure for honesty"
"value off >1%, unexplained",36,129,"rows kept; logged, value never adjusted"
entry count off by <=2,28,50,"rows kept; a one-or-two row gap (a total line or a footnote) is logged, not fixed by hand"
value within 1% (rounding),17,21,rows kept; filer rounding; logged
summary totals zero but holdings present (filer omitted totals),14,2,rows kept; no total to check against; logged
entry off by <=2 and value within 1%,4,21,rows kept; logged


In [15]:
# row-level, company-level and mapping problem classes, from the step reports
def grab(text, *keys):
    return [l.strip() for l in text.splitlines() if any(k in l for k in keys)]
for q in SPECS:
    print(f"=== {q} ===")
    print("shares/value check:            ", *grab(report(q, "shares_check.txt"), "FLAG RATE"))
    sc = report(q, "shares_check.txt"); a = sc.index("flagged rows by cause"); print(sc[a:sc.index("by parse path")].rstrip())
    print("CUSIP->CIK conflicts:          ", *grab(report(q, "cusip_cik_resolution.txt"), "conflicts, raw", "TWO-SIDED"), sep="\n    ")
    print("funds excluded / flags:")
    for l in grab(report(q, "ownership_report.txt"), "FUNDS EXCLUDED", "FIXES (Luke", "SHARE CLASSES", "NO COMPANY-LEVEL FIGURE", "companies over 100%", "large companies (top 500"):
        print("    " + l)
    rep = report(q, "ownership_report.txt"); a = rep.index("NO COMPANY-LEVEL FIGURE"); b = rep.index("largest without shares outstanding")
    print("    missing shares outstanding, by cause:\n" + "\n".join("      " + l for l in rep[a:b].splitlines()[2:]))
    print()

=== pilot_2013Q4 ===
shares/value check:             PASS 2 (scale-normalized internally): FLAG RATE 15,401 / 1,408,025 = 1.09%
flagged rows by cause (one per row, precedence as listed):
cause
unexplained                                                            7311
wrong shares column (swap note or >50% of filing flagged)              7141
scale (row-level x1000 / ÷1000)                                         428
cusip ambiguous (filing prices spread >tol: class / ADR / unit mix)     389
cusip repaired / short id                                               132
  sum 15,401
CUSIP->CIK conflicts:          
    conflicts, raw definition (>1 subject CIK): 220 = 3.61%  -> all flagged conflict_flag=1
    conflicts, TWO-SIDED definition (>=2 filings on each of >=2 CIKs): 27 = 0.44%   [stop condition 2%: not hit]
funds excluded / flags:
    FUNDS EXCLUDED (decision 2): 1,883 CUSIPs on 566 CIKs, 89,142 manager-CUSIP rows, $939.8bn of 13F value = 4.7% of the selected filings' value
    FIX

### Problem classes

| Problem | Handling |
|---|---|
| Summary page doesn't match | Kept and labeled; filer errors (dollars vs thousands, distinct-CUSIP counts) marked "parse right, summary wrong" |
| No rows parsed (some old text filings) | Listed by accession for the next fix round |
| Price check flags | Flagged with a cause; rate must not rise after a parser change |
| CUSIP mapped to two companies | Flagged; majority or closest 10-K/10-Q wins |
| Funds, debt CUSIPs, ADRs, duplicate amendments | Funds excluded; others flagged and left out of the figure |
| Share classes | Never summed; per-class figures |
| Missing shares outstanding | Cause recorded; Dr. Akcora's 2010 file fills the gap |
| Over 100% | Kept, flagged, cause recorded, not capped |